# Stage 1: Linear Regression (OLS) and Sensitivity Analysis
First stage of the model progression. One linear regression each for **NIV** and the **price spread** (PIMB − DA price), using a small set of physical inputs known at the **IDA1 run** (17:00 D-1). Each model is then put through sensitivity analysis to see which inputs drive its output, and which drive its extremes.

| Section | Content |
|---|---|
| 1 | Data for the IDA1 run (same leak-free rules as notebook 01) |
| 2 | Inputs and targets |
| 3 | OLS fit and test-set accuracy |
| 4 | Sensitivity analysis: Sobol indices on the output and on exceedance of P95/P99, fragility curves |

In [ ]:
import re
from masters_thesis.data.catalog import data      # all CSVs as DataFrames

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from SALib.analyze import sobol as sobol_analyze
from SALib.sample import sobol as sobol_sample

## 1. Data for the IDA1 run
Only the datasets used as inputs are loaded. The leak-free logic is the same as notebook 01, restricted to IDA1:
- **Forecasts** issued at or after their run time are removed (`tick_utc`).
- **Auction results** are only used from the next run (DA results are known at IDA1).
- **GB EPEX half-hourly DA** is published at 15:45 UTC on D-1, before the 17:00 IDA1 run.
- **Daily series** (EUR/GBP FX) count as published one day after their timestamp.

In [ ]:
START = "2022-01-31 23:00"   # start of trading day 2022-02-01 (Irish time)
RUN = "IDA1"
RUN_TIME = {"DA": (-1, "10:30"), "IDA1": (-1, "17:00")}        # (day offset, Irish clock time)
META_COLS = ["trading_day", "auction", "cutoff_utc", "tick_utc"]

DATASETS = [
    "ie_wind_meteologica_p50_mw_da", "ie_wind_meteologica_p50_mw_ida1",      # wind forecast, DA and IDA1 runs
    "ie_wind_meteologica_p10_mw_ida1", "ie_wind_meteologica_p90_mw_ida1",    # wind forecast uncertainty band
    "ie_demand_meteologica_p50_mw_da", "ie_demand_meteologica_p50_mw_ida1",  # demand forecast, DA and IDA1 runs
    "ie_da_mcv_wind_mw", "ie_da_mcv_bess_mw",                                # DA cleared wind and battery volumes
    "ie_da_actual_eur_hourly", "ie_da_actual_eur_half_hourly",              # DA price (half-hourly from Oct 2025)
    "gb_da_hh_actual_epex_gbp", "eur_gbp_daily_fx_eur",                     # GB half-hourly DA price, FX
]
FORECAST_RUN = {n: n.rsplit("_", 1)[1].upper() for n in DATASETS if n.endswith(("_da", "_ida1"))}
GB_PUBLISHED_UTC = "15:45"     # EPEX GB half-hourly DA results, D-1


def run_datetime(trading_day, run: str) -> pd.Series:
    """Run time (tz-aware Irish time) for each trading day."""
    days, clock = RUN_TIME[run]
    td = pd.to_datetime(pd.Series(trading_day))
    return (td + pd.Timedelta(days=days) + pd.Timedelta(clock + ":00")).dt.tz_localize("Europe/Dublin")


def values(df: pd.DataFrame) -> pd.Series:
    return df.drop(columns=META_COLS, errors="ignore").iloc[:, 0]


def to_half_hourly(s: pd.Series) -> pd.Series:
    """Half-hourly grid; hourly values fill their :30 slot."""
    step = pd.Timedelta(s.index.to_series().diff().mode().iloc[0])
    return s.resample("30min").ffill(limit=1) if step == pd.Timedelta(hours=1) else s.resample("30min").last()


def series_for(name: str) -> pd.Series:
    """One dataset on the half-hourly grid, forecasts issued at or after their run time removed."""
    df = data[name]
    s = values(df)
    if name in FORECAST_RUN:
        too_late = df["tick_utc"].values >= run_datetime(df["trading_day"], FORECAST_RUN[name]).values
        s = s.mask(too_late)
    return to_half_hourly(s)


fx = values(data["eur_gbp_daily_fx_eur"])                          # daily: handled after the trading day is known
d = pd.DataFrame({"niv": values(data["ie_niv_actual_mw"]), "pimb": values(data["ie_pimb_actual_eur"])})
d = d.join(pd.DataFrame({n: series_for(n) for n in DATASETS if n != "eur_gbp_daily_fx_eur"}))
d = d[d["niv"].notna() & d["pimb"].notna()].loc[START:].copy()

# Trading day D runs 23:00 D-1 to 23:00 D (Irish time)
local_date = d.index.normalize().tz_localize(None)
d["trading_day"] = local_date + pd.to_timedelta((d.index.hour == 23).astype("int64"), unit="D")
run_dt = pd.DatetimeIndex(run_datetime(d["trading_day"].values, RUN))

# GB half-hourly DA: blank on any day the run starts before publication
gb_pub = (pd.to_datetime(d["trading_day"]) - pd.Timedelta(days=1) + pd.Timedelta(GB_PUBLISHED_UTC + ":00")).dt.tz_localize("UTC")
d.loc[gb_pub.values >= run_dt.values, "gb_da_hh_actual_epex_gbp"] = np.nan

# FX: published one day after its timestamp; take the latest value published before the run time
fx_published = fx.copy()
fx_published.index = fx.index + pd.Timedelta(days=1)
d["eur_gbp_daily_fx_eur"] = fx_published.reindex(run_dt, method="ffill").values

print(f"{len(d):,} half-hours, {d['trading_day'].nunique():,} trading days")

In [ ]:
# Leakage check: every forecast value used was issued before the run, GB prices and FX published before it
leaks = {}
for name in FORECAST_RUN:
    df = data[name]
    tick = to_half_hourly(df["tick_utc"].where(series_for(name).reindex(df.index).notna())).reindex(d.index)
    leaks[name] = int(((tick.values >= run_dt.values) & d[name].notna().values).sum())
leaks["gb_da_hh_actual_epex_gbp"] = int(((gb_pub.values >= run_dt.values) & d["gb_da_hh_actual_epex_gbp"].notna().values).sum())
fx_pub_time = pd.Series(fx_published.index, index=fx_published.index).reindex(run_dt, method="ffill").values
leaks["eur_gbp_daily_fx_eur"] = int((fx_pub_time >= run_dt.values).sum())

leak_check = pd.Series(leaks, name="values used before available")
assert leak_check.sum() == 0, "Data leak detected"
leak_check

## 2. Inputs and targets
A small set of physical inputs, so the sensitivity analysis is interpretable. NBP gas is left out: it is strongly correlated with the DA price (r ≈ 0.73), and the sensitivity analysis below samples inputs independently, which would create unrealistic combinations of the two.

| Input | Definition | Why it may matter |
|---|---|---|
| `wind_forecast` | IE wind P50 forecast at IDA1 (MW) | Wind level and forecast-error exposure |
| `wind_revision` | IDA1 minus DA wind forecast (MW) | Wind change since the DA auction |
| `wind_uncertainty` | P90 − P10 wind band at IDA1 (MW) | How uncertain the wind forecast is |
| `wind_vs_da_cleared` | IDA1 wind forecast minus wind cleared in DA (MW) | Wind not yet positioned in the market |
| `demand_forecast` | IE demand P50 forecast at IDA1 (MW) | System load |
| `demand_revision` | IDA1 minus DA demand forecast (MW) | Demand change since the DA auction |
| `da_price` | DA cleared price (€/MWh) | Price level and system tightness |
| `gb_minus_ie_price` | GB DA price in € minus IE DA price (€/MWh) | Direction of interconnector flows |
| `da_bess` | Battery volume cleared in DA (MW) | Battery positions |
| `hour` | Hour of day (Irish time) | Daily pattern |

**Targets:** `niv` (MW, + = short system) and `spread` = PIMB − DA price (€/MWh).

In [ ]:
da_price = d["ie_da_actual_eur_half_hourly"].fillna(d["ie_da_actual_eur_hourly"])

X = pd.DataFrame({
    "wind_forecast":      d["ie_wind_meteologica_p50_mw_ida1"],
    "wind_revision":      d["ie_wind_meteologica_p50_mw_ida1"] - d["ie_wind_meteologica_p50_mw_da"],
    "wind_uncertainty":   (d["ie_wind_meteologica_p90_mw_ida1"] - d["ie_wind_meteologica_p10_mw_ida1"]).abs(),
    "wind_vs_da_cleared": d["ie_wind_meteologica_p50_mw_ida1"] - d["ie_da_mcv_wind_mw"],
    "demand_forecast":    d["ie_demand_meteologica_p50_mw_ida1"],
    "demand_revision":    d["ie_demand_meteologica_p50_mw_ida1"] - d["ie_demand_meteologica_p50_mw_da"],
    "da_price":           da_price,
    "gb_minus_ie_price":  d["gb_da_hh_actual_epex_gbp"] * d["eur_gbp_daily_fx_eur"] - da_price,
    "da_bess":            d["ie_da_mcv_bess_mw"],
    "hour":               d.index.hour + d.index.minute / 60,
})
Y = pd.DataFrame({"niv": d["niv"], "spread": d["pimb"] - da_price})

complete = X.notna().all(axis=1) & Y.notna().all(axis=1)
print(f"Complete rows: {complete.mean():.1%}")
X, Y, days = X[complete], Y[complete], d.loc[complete, "trading_day"]
X.describe().T.round(1)

In [ ]:
# Correlation between inputs: pairs above |0.6| make independent sampling in Section 4 less realistic
corr = X.corr()
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.index, zmin=-1, zmax=1,
                           colorscale="RdBu", text=corr.round(2).values, texttemplate="%{text}"))
fig.update_layout(title="Input correlations", height=550, width=700)
fig.show()

## 3. OLS fit and test-set accuracy
Inputs enter linearly, except hour of day, which enters through two sine/cosine pairs so the model can follow the daily shape. Standard errors are HAC (Newey-West), because errors in neighbouring half-hours are correlated. Chronological split: first 80% of trading days for fitting, last 20% for testing (same split day as notebooks 01 and 02).

In [ ]:
def design(X: pd.DataFrame) -> pd.DataFrame:
    """Regressors: linear inputs, two Fourier pairs for hour of day, and an intercept."""
    Z = X.drop(columns="hour").copy()
    angle = 2 * np.pi * X["hour"].to_numpy() / 24
    for k in (1, 2):
        Z[f"hour_sin{k}"], Z[f"hour_cos{k}"] = np.sin(k * angle), np.cos(k * angle)
    return sm.add_constant(Z, has_constant="add")


all_days = np.sort(days.unique())
split_day = all_days[int(len(all_days) * 0.8)]
train, test = (days < split_day).values, (days >= split_day).values
print(f"Train: before {pd.Timestamp(split_day):%Y-%m-%d}   Test: from {pd.Timestamp(split_day):%Y-%m-%d}")

fits = {t: sm.OLS(Y.loc[train, t], design(X[train])).fit(cov_type="HAC", cov_kwds={"maxlags": 48}) for t in Y}


def coef_table(fit) -> pd.DataFrame:
    out = pd.DataFrame({"coef": fit.params, "HAC s.e.": fit.bse, "p-value": fit.pvalues})
    out["effect of +1 s.d."] = out["coef"] * X[train].std().reindex(out.index)   # comparable across units
    return out


pd.concat({t: coef_table(f) for t, f in fits.items()}, axis=1).round(3)

In [ ]:
# Test-set accuracy against the zero benchmark (NIV = 0; spread = 0, i.e. PIMB = DA price)
def stats(y, pred, y_train):
    err = pred - y
    hi = y >= y_train.quantile(0.95)
    return {"MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean()), "Bias": err.mean(),
            "R²": 1 - (err ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "Upper-tail MAE (top 5%)": err[hi].abs().mean()}


predictions, rows = {}, {}
for t, fit in fits.items():
    y, y_train = Y.loc[test, t], Y.loc[train, t]
    predictions[t] = pd.Series(fit.predict(design(X[test])), index=y.index)
    rows[(t, "OLS")] = stats(y, predictions[t], y_train)
    rows[(t, "zero")] = stats(y, pd.Series(0.0, index=y.index), y_train)

results = pd.DataFrame(rows).T
results["MAE skill vs zero"] = 1 - results["MAE"] / results.xs("zero", level=1)["MAE"].reindex(results.index, level=0)
results.round(3)

In [ ]:
# Test set: actual vs OLS forecast (drag the slider or zoom to inspect)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=["NIV (MW)", "Spread: PIMB − DA (€/MWh)"])
for i, t in enumerate(["niv", "spread"], start=1):
    fig.add_trace(go.Scattergl(x=Y.index[test], y=Y.loc[test, t], name=f"Actual {t}",
                               line=dict(color="black", width=1)), row=i, col=1)
    fig.add_trace(go.Scattergl(x=predictions[t].index, y=predictions[t], name=f"OLS {t}",
                               line=dict(width=1)), row=i, col=1)
fig.update_layout(height=700, hovermode="x unified", title="Stage 1 OLS forecasts on the test set")
fig.show()

## 4. Sensitivity analysis
The fitted models are analysed on the training-period input distributions. Each input is sampled independently from its empirical distribution (via its quantile function), so the results describe the **model**, not the market, and ignore the input correlations shown in Section 2.

1. **Sobol indices on the output.** Share of the output variance due to each input. S1 = the input alone, ST = including interactions. For a linear model with independent inputs, S1 ≈ ST, and S1 can be checked exactly against b_i² Var(x_i) / Var(Y).
2. **Sobol indices on exceedance.** The output is replaced by 1 if it passes a threshold T, else 0. The indices then show which inputs decide whether an extreme happens, rather than the average. T is the P95 or P99 of the model's own output, so the event probability is fixed at 5% or 1%.
3. **Fragility curves.** Fix one input at a grid of values, resample the others, and plot P(output > T).

In [ ]:
INPUTS = list(X.columns)
X_train = X[train]
sorted_vals = {c: np.sort(X_train[c].to_numpy(dtype=float)) for c in INPUTS}


def from_uniform(u: np.ndarray) -> pd.DataFrame:
    """Uniform numbers in [0, 1] -> input values through each input's empirical quantile function."""
    u = np.atleast_2d(u)
    return pd.DataFrame({c: np.interp(u[:, j] * (len(v) - 1), np.arange(len(v)), v)
                         for j, (c, v) in enumerate(sorted_vals.items())})


def model(t: str):
    """Fitted OLS model for target t as a function of an input DataFrame."""
    beta = fits[t].params.to_numpy()
    return lambda Xs: design(Xs[INPUTS]).to_numpy() @ beta


PROBLEM = {"num_vars": len(INPUTS), "names": INPUTS, "bounds": [[0.0, 1.0]] * len(INPUTS)}


def sobol(t: str, threshold: float | None = None, n: int = 2 ** 14) -> pd.DataFrame:
    """Sobol S1 and ST (with 95% intervals) of the model output, or of the exceedance indicator."""
    u = sobol_sample.sample(PROBLEM, n, calc_second_order=False, seed=0)
    y = model(t)(from_uniform(u))
    if threshold is not None:
        y = (y > threshold).astype(float)
    res = sobol_analyze.analyze(PROBLEM, y, calc_second_order=False, seed=0)
    return pd.DataFrame({"S1": res["S1"], "S1 ±95%": res["S1_conf"], "ST": res["ST"], "ST ±95%": res["ST_conf"]},
                        index=INPUTS)


# Thresholds: P95 and P99 of each model's output under the input distributions
rng = np.random.default_rng(0)
u_mc = rng.uniform(0, 1, (200_000, len(INPUTS)))
thresholds = {t: {q: float(np.quantile(model(t)(from_uniform(u_mc)), q)) for q in (0.95, 0.99)} for t in Y}
pd.DataFrame({t: {"model P95": th[0.95], "model P99": th[0.99],
                  "actual P95 (train)": Y.loc[train, t].quantile(0.95),
                  "actual P99 (train)": Y.loc[train, t].quantile(0.99)} for t, th in thresholds.items()}).round(1)

In [ ]:
# 4.1 Sobol indices on the output, with the exact first-order values as a check
def exact_s1(t: str) -> pd.Series:
    """Exact S1 for this additive model: variance of each input's contribution, others held at the median."""
    u = (np.arange(20_000) + 0.5) / 20_000
    var = {}
    for j, c in enumerate(INPUTS):
        grid = np.full((len(u), len(INPUTS)), 0.5)
        grid[:, j] = u
        var[c] = np.var(model(t)(from_uniform(grid)))
    var = pd.Series(var)
    return var / var.sum()


sobol_output = {t: sobol(t).assign(**{"S1 exact": exact_s1(t)}) for t in Y}
pd.concat(sobol_output, axis=1).round(3)

In [ ]:
# 4.2 Sobol indices on exceedance of the P95 and P99 thresholds (total-order ST)
sobol_exceed = {(t, f"P{int(q * 100)}"): sobol(t, thresholds[t][q]) for t in Y for q in (0.95, 0.99)}
st_table = pd.concat({t: pd.DataFrame({"output": sobol_output[t]["ST"],
                                       "exceed P95": sobol_exceed[(t, "P95")]["ST"],
                                       "exceed P99": sobol_exceed[(t, "P99")]["ST"]}) for t in Y}, axis=1)
st_table.round(3)

In [ ]:
# Total-order indices: output vs exceedance, per target
fig = make_subplots(rows=1, cols=2, subplot_titles=["NIV", "Spread (PIMB − DA)"], shared_yaxes=True)
for i, t in enumerate(["niv", "spread"], start=1):
    for col in st_table[t].columns:
        fig.add_trace(go.Bar(y=INPUTS, x=st_table[t][col], orientation="h", name=col,
                             legendgroup=col, showlegend=(i == 1)), row=1, col=i)
fig.update_layout(barmode="group", height=550, title="Total-order Sobol indices: average output vs exceedance",
                  xaxis_title="ST", xaxis2_title="ST")
fig.update_yaxes(autorange="reversed")
fig.show()

In [ ]:
# 4.3 Fragility curves: P(output > model P99 | input fixed), other inputs resampled
GRID = np.linspace(0.01, 0.99, 25)       # 1st to 99th percentile of each input
N_COND = 20_000


def fragility(t: str, threshold: float) -> pd.DataFrame:
    rows = []
    for j, c in enumerate(INPUTS):
        u = np.random.default_rng(j).uniform(0, 1, (N_COND, len(INPUTS)))
        for g in GRID:
            u[:, j] = g
            rows.append({"input": c, "x": from_uniform(np.full((1, len(INPUTS)), g))[c].iloc[0],
                         "pf": (model(t)(from_uniform(u)) > threshold).mean()})
    return pd.DataFrame(rows)


curves = {t: fragility(t, thresholds[t][0.99]) for t in Y}

for t, cv in curves.items():
    fig = make_subplots(rows=2, cols=5, subplot_titles=INPUTS, vertical_spacing=0.15)
    for k, c in enumerate(INPUTS):
        sub = cv[cv["input"] == c]
        fig.add_trace(go.Scatter(x=sub["x"], y=sub["pf"], mode="lines+markers", marker=dict(size=4),
                                 showlegend=False), row=k // 5 + 1, col=k % 5 + 1)
        fig.add_hline(y=0.01, line=dict(color="red", dash="dot", width=1), row=k // 5 + 1, col=k % 5 + 1)
    fig.update_layout(height=550, title=f"{t}: P(output > model P99 | input), red line = unconditional 1%")
    fig.show()

# Importance: range of each curve, normalised to sum to 1
fragility_importance = pd.DataFrame({t: cv.groupby("input")["pf"].agg(lambda p: p.max() - p.min())
                                     for t, cv in curves.items()}).reindex(INPUTS)
(fragility_importance / fragility_importance.sum()).round(3)

## 5. Notes
- **What the model can reach.** Compare the model P95/P99 thresholds with the actual P95/P99 in Section 4. A model of the conditional mean is smooth, so its own "extremes" sit well inside the real ones. The exceedance analysis therefore describes what pushes this model towards its extremes, not what causes real spikes. Quantile regression (stage 2) models the tail directly.
- **Correlated inputs.** All indices assume independent inputs. Inputs correlated above |0.6| should be read with care.
- **Next stage:** quantile regression at P50, P90, P95 and P99 with the same inputs, split and sensitivity analysis.